In [1]:
import torch
from monai.utils import optional_import
from monai.data.box_utils import StandardMode, get_spatial_dims
from monai.apps.detection.transforms.array import ConvertBoxToStandardMode
import torch_geometric.data as data

# convert boxes to points coord
def boxes_to_coord(boxes, spatial_size, mode=StandardMode):
    center = []
    for i in (spatial_size):
        center.append(i // 2)
    box_converter = ConvertBoxToStandardMode(mode=mode)
    boxes_t = box_converter(boxes)
    spatial_dims = get_spatial_dims(boxes=boxes_t)
    if spatial_dims == 2:
        x1, y1, x2, y2 = boxes.split(1, dim=-1)
        coords = torch.tensor([[x1, y1], [x1, y2], [x2, y1], [x2, y2]])
    elif spatial_dims == 3:
        x1, y1, z1, x2, y2, z2 = boxes.split(1, dim=-1)
        coords = torch.tensor(
            [
                [x1, y1, z1], [x1, y2, z1], [x2, y1, z1], [x2, y2, z1],
                [x1, y1, z2], [x1, y2, z2], [x2, y1, z2], [x2, y2, z2]
            ])
    coords -= torch.tensor(center)
    return coords

[1703855891.023691] [yunliu-ms-7d31:1093835:f]        vfs_fuse.c:281  UCX  ERROR inotify_add_watch(/tmp) failed: No space left on device


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Geometric:
    """
    This is a wrapper transform for PyTorch Geometric transform based on the specified transform name and args.
    """

    def __init__(self, name: str, *args, **kwargs) -> None:
        """
        Args:
            name: The transform name in Geometric package.
            args: parameters for the Geometric transform.
            kwargs: parameters for the Geometric transform.

        """
        super().__init__()
        self.name = name
        transform, _ = optional_import("torch_geometric.transforms", name=name)
        self.trans = transform(*args, **kwargs)

    def __call__(self, pos, spatial_size):
        center = []
        for i in (spatial_size):
            center.append(i // 2)
        data_geometric = data.Data(pos=pos)

        out = self.trans(data_geometric)
        out_geometric = out.pos
        out_geometric += torch.tensor(center)
        return out_geometric

In [3]:
boxes = torch.tensor([[1, 3, 3, 7]]) # StandardMode: xyxy

pos = boxes_to_coord(boxes=boxes, spatial_size=[10, 10])
Geometric("RandomFlip", axis=1, p=1)(pos=pos, spatial_size=[10, 10])

tensor([[1, 7],
        [1, 3],
        [3, 7],
        [3, 3]])

In [4]:
import torch_geometric.transforms as T
T.RandomScale(scales=[0.5, 0.5])

RandomScale([0.5, 0.5])

In [5]:
Geometric("RandomRotate", degrees=[90, 90], axis=1)(pos=pos, spatial_size=[10, 10])

tensor([[3, 9],
        [7, 9],
        [3, 7],
        [7, 7]])

In [6]:
Geometric("RandomScale", scales=[0.5, 0.5])(pos=pos, spatial_size=[10, 10])

tensor([[3., 4.],
        [3., 6.],
        [4., 4.],
        [4., 6.]])

In [9]:
from monai.apps.detection.transforms.array import FlipBox, RotateBox90, ZoomBox

trans_monai = FlipBox(spatial_axis=1)
trans_monai(boxes=boxes, spatial_size=(10, 10))

tensor([[1, 3, 3, 7]])

In [8]:
trans_monai = RotateBox90(spatial_axes=(1, 0))
trans_monai(boxes=boxes, spatial_size=(10, 10))

tensor([[3, 7, 7, 9]])

In [11]:
trans_monai = ZoomBox(zoom=0.5)
trans_monai(boxes=boxes)

tensor([[0, 1, 1, 3]])